In [16]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv')

base = ['engine_displacement','horsepower','vehicle_weight','model_year']

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    
    return w[0], w[1:]

def prepare_X(df, fill_num=0):
    df_num = df[base].copy()
    df_num = df_num.fillna(fill_num)
    X = df_num.values
    return X

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)


scores = []
for s in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(s)
    
    n = len(df)
    n_val = int(0.2 * n)
    n_test = int(0.2 * n)
    n_train = n - (n_val + n_test)
    
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_shuffled = df[base + ['fuel_efficiency_mpg']].iloc[idx]
    
    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()

    # The issue before was I was taking the log which was unnecessary because there's no long tail!
    
    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values
    
    X_train = prepare_X(df_train, fill_num=0)
    X_val = prepare_X(df_val, fill_num=0)
    
    w_0, w = train_linear_regression(X_train, y_train)
    y_pred = w_0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    
    scores.append(score)
    print(f'Seed {s}: RMSE = {score:.6f}')

print(f'\nStandard deviation: {round(np.std(scores), 3)}')
np.std(scores)

Seed 0: RMSE = 0.520653
Seed 1: RMSE = 0.521339
Seed 2: RMSE = 0.522807
Seed 3: RMSE = 0.515952
Seed 4: RMSE = 0.510913
Seed 5: RMSE = 0.528341
Seed 6: RMSE = 0.531391
Seed 7: RMSE = 0.509067
Seed 8: RMSE = 0.514740
Seed 9: RMSE = 0.513187

Standard deviation: 0.007


0.006989446426422696

In [18]:
np.random.seed(9)

n = len(df)

n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - (n_val + n_test)

idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df[base + ['fuel_efficiency_mpg']].iloc[idx]  # Include target

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
df_test = df_shuffled.iloc[n_train+n_val:].copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove target from feature dataframes
df_train_features = df_train.drop('fuel_efficiency_mpg', axis=1)
df_val_features = df_val.drop('fuel_efficiency_mpg', axis=1)
df_test_features = df_test.drop('fuel_efficiency_mpg', axis=1)

# Combine train and validation
df_full_train = pd.concat([df_train_features, df_val_features])
df_full_train = df_full_train.reset_index(drop=True)

y_full_train = np.concatenate([y_train, y_val])  # Use new y values!

# Prepare data and train
X_full_train = prepare_X(df_full_train, fill_num=0)  # Fill with 0
w_0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

# Evaluate on test set
X_test = prepare_X(df_test_features, fill_num=0)  # Fill with 0
y_pred = w_0 + X_test.dot(w)

score = rmse(y_test, y_pred)  # Use new y_test!

print(f"RMSE on test set: {round(score, 2)}")

RMSE on test set: 0.52
